# SAM3 TTD Few-Shot Experiment Runner

Run the **Experiment Selection** cell first. The notebook supports two workflows:

1. Single experiment: set `EXPERIMENT_KEY`, then call `run_train(EXPERIMENT_KEY)` / `run_eval(EXPERIMENT_KEY)` from the helper cell.
2. Batch experiment: run the **Batch Train + Eval** cell to process all stable 5 / 10 / 25 / 50-shot experiments for `Single-TB`, `Shift-TA_TC-to-TB_10pct`, and `Shift-TA_TB-to-TC_10pct`.

The batch order intentionally runs all `Single-TB` YAMLs first. This matches the safer ordering used after the earlier dtype / gradient issues.


In [ ]:
# Experiment Selection
import os
from pathlib import Path


def find_project_repo_root(start=None):
    """Find the CSCI5527-final repo root from the current notebook working directory."""
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "SAM3").exists() and (candidate / "TACK_Tunnel_Data").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the CSCI5527-final repo root. "
        "Start Jupyter inside the project repo, or update this helper."
    )


PROJECT_REPO_ROOT = find_project_repo_root()
WORKSPACE_ROOT = PROJECT_REPO_ROOT.parent
PROJECT_ROOT = WORKSPACE_ROOT  # Backward-compatible name used below.
SAM3_WORK_ROOT = PROJECT_REPO_ROOT / "SAM3"
SAM3_REPO_ROOT = Path(os.environ.get("SAM3_REPO_ROOT", WORKSPACE_ROOT / "sam3")).resolve()

if not SAM3_REPO_ROOT.exists():
    raise FileNotFoundError(
        f"SAM3 repo not found: {SAM3_REPO_ROOT}\n"
        "Expected layout: <workspace>/CSCI5527-final and <workspace>/sam3.\n"
        "If SAM3 is elsewhere, set os.environ['SAM3_REPO_ROOT'] before running this cell."
    )

# Experiment key constants. These let you write EXPERIMENT_KEY without quotes.
single_tb_5shot_stable_lowlr_v1 = "single_tb_5shot_stable_lowlr_v1"
single_tb_10shot_stable_lowlr_v1 = "single_tb_10shot_stable_lowlr_v1"
single_tb_25shot_stable_lowlr_v1 = "single_tb_25shot_stable_lowlr_v1"
single_tb_50shot_stable_lowlr_v1 = "single_tb_50shot_stable_lowlr_v1"
shift_ta_tc_to_tb_10pct_5shot_stable_lowlr_v1 = "shift_ta_tc_to_tb_10pct_5shot_stable_lowlr_v1"
shift_ta_tc_to_tb_10pct_10shot_stable_lowlr_v1 = "shift_ta_tc_to_tb_10pct_10shot_stable_lowlr_v1"
shift_ta_tc_to_tb_10pct_25shot_stable_lowlr_v1 = "shift_ta_tc_to_tb_10pct_25shot_stable_lowlr_v1"
shift_ta_tc_to_tb_10pct_50shot_stable_lowlr_v1 = "shift_ta_tc_to_tb_10pct_50shot_stable_lowlr_v1"
shift_ta_tb_to_tc_10pct_5shot_stable_lowlr_v1 = "shift_ta_tb_to_tc_10pct_5shot_stable_lowlr_v1"
shift_ta_tb_to_tc_10pct_10shot_stable_lowlr_v1 = "shift_ta_tb_to_tc_10pct_10shot_stable_lowlr_v1"
shift_ta_tb_to_tc_10pct_25shot_stable_lowlr_v1 = "shift_ta_tb_to_tc_10pct_25shot_stable_lowlr_v1"
shift_ta_tb_to_tc_10pct_50shot_stable_lowlr_v1 = "shift_ta_tb_to_tc_10pct_50shot_stable_lowlr_v1"
single_tb_5shot_clean_fp32_v1 = "single_tb_5shot_clean_fp32_v1"
single_tb_10shot_clean_fp32_v1 = "single_tb_10shot_clean_fp32_v1"
single_tb_25shot_clean_fp32_v1 = "single_tb_25shot_clean_fp32_v1"
single_tb_50shot_clean_fp32_v1 = "single_tb_50shot_clean_fp32_v1"
shift_ta_tc_to_tb_10pct_5shot_clean_fp32_v1 = "shift_ta_tc_to_tb_10pct_5shot_clean_fp32_v1"
shift_ta_tc_to_tb_10pct_10shot_clean_fp32_v1 = "shift_ta_tc_to_tb_10pct_10shot_clean_fp32_v1"
shift_ta_tc_to_tb_10pct_25shot_clean_fp32_v1 = "shift_ta_tc_to_tb_10pct_25shot_clean_fp32_v1"
shift_ta_tc_to_tb_10pct_50shot_clean_fp32_v1 = "shift_ta_tc_to_tb_10pct_50shot_clean_fp32_v1"
shift_ta_tb_to_tc_10pct_5shot_clean_fp32_v1 = "shift_ta_tb_to_tc_10pct_5shot_clean_fp32_v1"
shift_ta_tb_to_tc_10pct_10shot_clean_fp32_v1 = "shift_ta_tb_to_tc_10pct_10shot_clean_fp32_v1"
shift_ta_tb_to_tc_10pct_25shot_clean_fp32_v1 = "shift_ta_tb_to_tc_10pct_25shot_clean_fp32_v1"
shift_ta_tb_to_tc_10pct_50shot_clean_fp32_v1 = "shift_ta_tb_to_tc_10pct_50shot_clean_fp32_v1"

# Single-experiment default. Change this when you want to inspect one run with the IoU cell.
EXPERIMENT_KEY = single_tb_5shot_stable_lowlr_v1

EXPERIMENT_GROUPS = {
    "single_tb": "Single-TB",
    "shift_ta_tc_to_tb_10pct": "Shift-TA_TC-to-TB_10pct",
    "shift_ta_tb_to_tc_10pct": "Shift-TA_TB-to-TC_10pct",
}

# Important: keep Single-TB first. Running these YAMLs first avoids the earlier dtype/gradient issue.
BATCH_GROUP_ORDER = [
    "single_tb",
    "shift_ta_tc_to_tb_10pct",
    "shift_ta_tb_to_tc_10pct",
]
BATCH_SHOTS = [5, 10, 25, 50]
BATCH_RUN_TAG = "stable_lowlr_v1"

EXPERIMENTS = {}

def register_experiment(group_key, shot_per_class, run_tag):
    experiment_name = EXPERIMENT_GROUPS[group_key]
    config_stem = f"ttd_{group_key}_{shot_per_class}shot_textseg_{run_tag}"
    run_root = SAM3_WORK_ROOT / "fewshot_data" / experiment_name / f"{shot_per_class}_shot_per_class"
    run_dir = SAM3_WORK_ROOT / "outputs" / "fewshot_runs" / experiment_name / f"{shot_per_class}_shot_per_class_{run_tag}"
    key = f"{group_key}_{shot_per_class}shot_{run_tag}"
    EXPERIMENTS[key] = {
        "key": key,
        "experiment_name": experiment_name,
        "group_key": group_key,
        "shot_per_class": shot_per_class,
        "run_tag": run_tag,
        "run_root": run_root,
        "run_dir": run_dir,
        "train_config": f"configs/ttd_fewshot/{config_stem}.yaml",
        "test_config": f"configs/ttd_fewshot/{config_stem}_test_eval.yaml",
    }

for group_key in EXPERIMENT_GROUPS:
    for shot_per_class in (5, 10, 25, 50):
        register_experiment(group_key, shot_per_class, "stable_lowlr_v1")
        register_experiment(group_key, shot_per_class, "clean_fp32_v1")

BATCH_EXPERIMENT_KEYS = [
    f"{group_key}_{shot_per_class}shot_{BATCH_RUN_TAG}"
    for group_key in BATCH_GROUP_ORDER
    for shot_per_class in BATCH_SHOTS
]

def set_experiment(experiment_key):
    if experiment_key not in EXPERIMENTS:
        available = "\n".join(f"  - {key}" for key in sorted(EXPERIMENTS))
        raise KeyError(f"Unknown EXPERIMENT_KEY: {experiment_key}\nAvailable keys:\n{available}")

    experiment = EXPERIMENTS[experiment_key]
    train_config = experiment["train_config"]
    test_config = experiment["test_config"]
    run_root = experiment["run_root"]
    run_dir = experiment["run_dir"]
    checkpoint = run_dir / "checkpoints" / "checkpoint.pt"
    gt_json = run_root / "test" / "_annotations.coco.json"
    pred_json = run_dir / "dumps" / "ttd" / "test" / "coco_predictions_segm.json"

    train_config_path = SAM3_REPO_ROOT / "sam3" / "train" / train_config
    test_config_path = SAM3_REPO_ROOT / "sam3" / "train" / test_config
    for label, path in {
        "train config": train_config_path,
        "test config": test_config_path,
        "dataset root": run_root,
        "test ground truth": gt_json,
    }.items():
        if not path.exists():
            raise FileNotFoundError(f"Missing {label}: {path}")

    globals().update({
        "EXPERIMENT_KEY": experiment_key,
        "EXPERIMENT": experiment,
        "TRAIN_CONFIG": train_config,
        "TEST_CONFIG": test_config,
        "RUN_ROOT": run_root,
        "RUN_DIR": run_dir,
        "CHECKPOINT": checkpoint,
        "GT_JSON": gt_json,
        "PRED_JSON": pred_json,
        "TRAIN_CONFIG_PATH": train_config_path,
        "TEST_CONFIG_PATH": test_config_path,
    })
    return experiment

EXPERIMENT = set_experiment(EXPERIMENT_KEY)

print("Selected experiment:", EXPERIMENT_KEY)
print("Experiment name:", EXPERIMENT["experiment_name"])
print("Shot per class:", EXPERIMENT["shot_per_class"])
print("Run tag:", EXPERIMENT["run_tag"])
print("Training config:", TRAIN_CONFIG)
print("Test eval config:", TEST_CONFIG)
print("Dataset root:", RUN_ROOT)
print("Output dir:", RUN_DIR)
print("\nBatch run order:")
for idx, key in enumerate(BATCH_EXPERIMENT_KEYS, start=1):
    exp = EXPERIMENTS[key]
    print(f"{idx:02d}. {key} -> {exp['experiment_name']} / {exp['shot_per_class']}-shot")


In [ ]:
# Train/Eval Helpers
import os
import subprocess
import sys
from datetime import datetime

if "EXPERIMENTS" not in globals():
    raise RuntimeError("Run the Experiment Selection cell first.")

RUN_ENV = os.environ.copy()
RUN_ENV.setdefault("HYDRA_FULL_ERROR", "1")

def _train_cmd(config):
    return [
        sys.executable,
        "-m", "sam3.train.train",
        "-c", config,
        "--use-cluster", "0",
        "--num-gpus", "1",
    ]

def _run_sam3_command(cmd, label, experiment_key):
    print("=" * 100)
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {label}: {experiment_key}")
    print("Command:", " ".join(cmd))
    return subprocess.run(cmd, check=True, cwd=str(SAM3_REPO_ROOT), env=RUN_ENV)

def run_train(experiment_key=None, skip_existing=True):
    experiment_key = experiment_key or EXPERIMENT_KEY
    experiment = set_experiment(experiment_key)
    print("Selected experiment:", experiment_key)
    print("Training config:", TRAIN_CONFIG)
    print("Output dir:", RUN_DIR)

    if skip_existing and CHECKPOINT.exists():
        print("Skipping train because checkpoint already exists:", CHECKPOINT)
        return "skipped"

    _run_sam3_command(_train_cmd(TRAIN_CONFIG), "TRAIN", experiment_key)
    print("Checkpoint:", CHECKPOINT)
    print("Checkpoint exists:", CHECKPOINT.exists())
    return "trained"

def run_eval(experiment_key=None, skip_existing=True):
    experiment_key = experiment_key or EXPERIMENT_KEY
    experiment = set_experiment(experiment_key)
    print("Selected experiment:", experiment_key)
    print("Test eval config:", TEST_CONFIG)
    print("Using checkpoint:", CHECKPOINT)

    if not CHECKPOINT.exists():
        raise FileNotFoundError(f"Train checkpoint not found yet: {CHECKPOINT}")
    if skip_existing and PRED_JSON.exists() and PRED_JSON.stat().st_size > 2:
        print("Skipping eval because prediction file already exists:", PRED_JSON)
        return "skipped"

    _run_sam3_command(_train_cmd(TEST_CONFIG), "TEST EVAL", experiment_key)
    print("Prediction file:", PRED_JSON)
    print("Prediction exists:", PRED_JSON.exists())
    return "evaluated"

def run_train_and_eval_many(
    experiment_keys,
    do_train=True,
    do_eval=True,
    skip_existing_train=True,
    skip_existing_eval=True,
    stop_on_error=True,
):
    results = []
    for idx, experiment_key in enumerate(experiment_keys, start=1):
        experiment = EXPERIMENTS[experiment_key]
        print("\n" + "#" * 100)
        print(
            f"Batch item {idx}/{len(experiment_keys)}: {experiment_key} "
            f"({experiment['experiment_name']}, {experiment['shot_per_class']}-shot)"
        )
        train_status = "not_requested"
        eval_status = "not_requested"
        error = None
        try:
            if do_train:
                train_status = run_train(experiment_key, skip_existing=skip_existing_train)
            if do_eval:
                eval_status = run_eval(experiment_key, skip_existing=skip_existing_eval)
        except Exception as exc:
            error = repr(exc)
            print(f"ERROR in {experiment_key}: {error}")
            if stop_on_error:
                raise
        results.append({
            "experiment_key": experiment_key,
            "experiment_name": experiment["experiment_name"],
            "shot_per_class": experiment["shot_per_class"],
            "run_tag": experiment["run_tag"],
            "train_status": train_status,
            "eval_status": eval_status,
            "checkpoint": str(CHECKPOINT),
            "prediction_json": str(PRED_JSON),
            "error": error,
        })
    return results


In [ ]:
# Batch Train + Eval: all stable few-shot experiments
# This runs Single-TB first, then the two domain-shift combinations.
# Set SKIP_EXISTING_* to False if you want to force reruns.

if "BATCH_EXPERIMENT_KEYS" not in globals():
    raise RuntimeError("Run the Experiment Selection cell first.")
if "run_train_and_eval_many" not in globals():
    raise RuntimeError("Run the Train/Eval Helpers cell first.")

BATCH_DO_TRAIN = True
BATCH_DO_EVAL = True
SKIP_EXISTING_TRAIN = True
SKIP_EXISTING_EVAL = True
STOP_ON_ERROR = True

batch_results = run_train_and_eval_many(
    BATCH_EXPERIMENT_KEYS,
    do_train=BATCH_DO_TRAIN,
    do_eval=BATCH_DO_EVAL,
    skip_existing_train=SKIP_EXISTING_TRAIN,
    skip_existing_eval=SKIP_EXISTING_EVAL,
    stop_on_error=STOP_ON_ERROR,
)

try:
    import pandas as pd
    display(pd.DataFrame(batch_results))
except Exception:
    for row in batch_results:
        print(row)


## IoU Analysis: How to Read It

COCO AP already uses IoU internally, but it does not show direct overlap statistics. This cell adds a more literal mask-overlap view for the test set.

- **mean_best_iou_gt**: for each ground-truth crack mask, find the best overlapping predicted mask; higher means masks overlap the objects better.
- **gt_recall@0.50 / gt_recall@0.75**: fraction of ground-truth objects that have at least one prediction with mask IoU above the threshold. This answers, “how many real cracks did we find?”
- **pred_precision@0.50 / pred_precision@0.75**: fraction of predictions that match a ground-truth object above the threshold. This answers, “how many predicted cracks were credible?”
- **f1@threshold**: harmonic mean of recall and precision at that IoU threshold.
- **matched_mean_iou@threshold**: mean IoU of one-to-one greedy matches at that threshold; this ignores unmatched false positives/false negatives, so read it together with recall and precision.

A common interpretation is: IoU@0.50 is a loose localization check, IoU@0.75 is a stricter mask-quality check. If recall@0.50 is much higher than recall@0.75, the model is roughly finding objects but mask boundaries/localization are weak.

The score-threshold table is important because the prediction dump keeps many low-confidence masks. Very low precision at `score >= 0.00` can simply mean there are many low-confidence extra masks; check whether precision improves as the score threshold increases.


In [ ]:
# IoU Analysis: direct mask-overlap metrics on the test predictions
import numpy as np
import pandas as pd
from pycocotools.coco import COCO
from pycocotools import mask as mask_utils

if "EXPERIMENT" not in globals():
    raise RuntimeError("Run the Experiment Selection cell first.")

PRED_JSON = RUN_DIR / "dumps" / "ttd" / "test" / "coco_predictions_segm.json"
GT_JSON = RUN_ROOT / "test" / "_annotations.coco.json"

print("Selected experiment:", EXPERIMENT_KEY)
print("Prediction file:", PRED_JSON)
print("Ground truth file:", GT_JSON)

if not PRED_JSON.exists():
    raise FileNotFoundError(f"Prediction file not found. Run the Test Eval cell first: {PRED_JSON}")
if PRED_JSON.stat().st_size <= 2:
    raise ValueError(f"Prediction file is empty. Re-run Test Eval after the config fixes: {PRED_JSON}")

coco_gt = COCO(str(GT_JSON))
coco_dt = coco_gt.loadRes(str(PRED_JSON))

cat_ids = coco_gt.getCatIds()
img_ids = coco_gt.getImgIds()
cat_name = {cat["id"]: cat["name"] for cat in coco_gt.loadCats(cat_ids)}

def ann_to_rle(coco, ann):
    """Return a COCO RLE dict for one annotation segmentation."""
    h, w = coco.imgs[ann["image_id"]]["height"], coco.imgs[ann["image_id"]]["width"]
    seg = ann["segmentation"]
    if isinstance(seg, list):
        rles = mask_utils.frPyObjects(seg, h, w)
        return mask_utils.merge(rles)
    if isinstance(seg.get("counts"), list):
        return mask_utils.frPyObjects(seg, h, w)
    return seg

def greedy_match_ious(iou_matrix, threshold):
    """Greedy one-to-one matching by descending IoU."""
    if iou_matrix.size == 0:
        return []
    pairs = []
    for pred_idx, gt_idx in zip(*np.where(iou_matrix >= threshold)):
        pairs.append((float(iou_matrix[pred_idx, gt_idx]), int(pred_idx), int(gt_idx)))
    pairs.sort(reverse=True)
    used_pred, used_gt, matched = set(), set(), []
    for iou, pred_idx, gt_idx in pairs:
        if pred_idx in used_pred or gt_idx in used_gt:
            continue
        used_pred.add(pred_idx)
        used_gt.add(gt_idx)
        matched.append(iou)
    return matched

def summarize_iou(cat_id=None, score_threshold=0.0, thresholds=(0.50, 0.75)):
    best_iou_per_gt = []
    best_iou_per_pred = []
    matched_by_threshold = {thr: [] for thr in thresholds}
    total_gt = 0
    total_pred = 0
    images_with_gt = 0
    images_with_pred = 0

    cats = [cat_id] if cat_id is not None else cat_ids
    for img_id in img_ids:
        for cur_cat_id in cats:
            gt_anns = coco_gt.loadAnns(coco_gt.getAnnIds(imgIds=[img_id], catIds=[cur_cat_id], iscrowd=None))
            dt_anns = coco_dt.loadAnns(coco_dt.getAnnIds(imgIds=[img_id], catIds=[cur_cat_id]))
            dt_anns = [ann for ann in dt_anns if ann.get("score", 0.0) >= score_threshold]
            dt_anns = sorted(dt_anns, key=lambda ann: ann.get("score", 0.0), reverse=True)

            if gt_anns:
                images_with_gt += 1
            if dt_anns:
                images_with_pred += 1
            total_gt += len(gt_anns)
            total_pred += len(dt_anns)

            if not gt_anns and not dt_anns:
                continue
            if not gt_anns:
                best_iou_per_pred.extend([0.0] * len(dt_anns))
                continue
            if not dt_anns:
                best_iou_per_gt.extend([0.0] * len(gt_anns))
                continue

            gt_rles = [ann_to_rle(coco_gt, ann) for ann in gt_anns]
            dt_rles = [ann["segmentation"] for ann in dt_anns]
            iscrowd = [int(ann.get("iscrowd", 0)) for ann in gt_anns]
            ious = mask_utils.iou(dt_rles, gt_rles, iscrowd)

            best_iou_per_gt.extend(ious.max(axis=0).tolist())
            best_iou_per_pred.extend(ious.max(axis=1).tolist())
            for thr in thresholds:
                matched_by_threshold[thr].extend(greedy_match_ious(ious, thr))

    best_iou_per_gt = np.asarray(best_iou_per_gt, dtype=float)
    best_iou_per_pred = np.asarray(best_iou_per_pred, dtype=float)

    row = {
        "category": "all" if cat_id is None else cat_name.get(cat_id, str(cat_id)),
        "score_threshold": score_threshold,
        "images": len(img_ids),
        "images_with_gt": images_with_gt,
        "images_with_pred": images_with_pred,
        "gt_instances": total_gt,
        "pred_instances": total_pred,
        "mean_best_iou_gt": float(best_iou_per_gt.mean()) if total_gt else np.nan,
        "median_best_iou_gt": float(np.median(best_iou_per_gt)) if total_gt else np.nan,
        "mean_best_iou_pred": float(best_iou_per_pred.mean()) if total_pred else np.nan,
    }

    for thr in thresholds:
        matched = matched_by_threshold[thr]
        recall = len(matched) / total_gt if total_gt else np.nan
        precision = len(matched) / total_pred if total_pred else np.nan
        f1 = (2 * precision * recall / (precision + recall)) if precision + recall > 0 else 0.0
        row[f"gt_recall@{thr:.2f}"] = recall
        row[f"pred_precision@{thr:.2f}"] = precision
        row[f"f1@{thr:.2f}"] = f1
        row[f"matched_mean_iou@{thr:.2f}"] = float(np.mean(matched)) if matched else 0.0

    return row, best_iou_per_gt, best_iou_per_pred

def rounded_df(rows):
    df = pd.DataFrame(rows)
    for col in df.columns:
        if df[col].dtype.kind in "fc":
            df[col] = df[col].round(4)
    return df

print(f"Prediction file: {PRED_JSON}")
print(f"Ground truth file: {GT_JSON}")

rows = []
overall_row, best_gt, best_pred = summarize_iou(cat_id=None, score_threshold=0.0)
rows.append(overall_row)
for cat_id in cat_ids:
    row, _, _ = summarize_iou(cat_id=cat_id, score_threshold=0.0)
    rows.append(row)

print("Overall/category IoU summary using all predictions:")
display(rounded_df(rows))

threshold_rows = []
for score_thr in [0.0, 0.01, 0.05, 0.10, 0.25, 0.50]:
    row, _, _ = summarize_iou(cat_id=None, score_threshold=score_thr)
    threshold_rows.append(row)

print("IoU summary at different prediction score thresholds:")
display(rounded_df(threshold_rows))

hist_bins = [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]
hist_counts, bin_edges = np.histogram(best_gt, bins=hist_bins)
hist_df = pd.DataFrame({
    "best_gt_iou_bin": [f"[{bin_edges[i]:.2f}, {bin_edges[i+1]:.2f})" for i in range(len(hist_counts))],
    "gt_count": hist_counts,
})
hist_df.loc[len(hist_df) - 1, "best_gt_iou_bin"] = f"[{bin_edges[-2]:.2f}, {bin_edges[-1]:.2f}]"
print("Best-IoU distribution over ground-truth instances:")
display(hist_df)
